# nanlmv9 Mamba/MoE ablation smoke

GPU shape/step smoke for the 256-expert MoE + selective-SSM ablations.
This is a few-minute run: it validates every architecture variant and writes
`ablation.json`; it is not a converged language model.

In [ ]:
!git clone -q https://github.com/shemshallah/nanlmv9.git
%cd nanlmv9
import torch, os
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
!pip -q install --no-cache-dir pydivsufsort || true


In [ ]:
!mkdir -p data
!curl -sSL -o data/wt2_train.txt https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/train.txt
!curl -sSL -o data/wt2_valid.txt https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/valid.txt
!curl -sSL -o data/wt2_test.txt https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/test.txt
!curl -sSL -o data/wordlist.txt https://raw.githubusercontent.com/first20hours/google-10000-english/master/google-10000-english.txt
!wc -c data/*.txt

In [ ]:
!python3 experiments/mamba_moe/tokenizer_bpe.py   --vocab-texts data/wt2_train.txt data/wordlist.txt   --train-texts data/wt2_train.txt   --valid-texts data/wt2_valid.txt   --test-texts data/wt2_test.txt   --out data/tok32k --vocab-size 32000

In [ ]:
!python3 experiments/mamba_moe/tokenizer.py \
  --vocab-texts data/wt2_train.txt data/wordlist.txt \
  --train-texts data/wt2_train.txt \
  --valid-texts data/wt2_valid.txt \
  --test-texts data/wt2_test.txt \
  --out data/tok32k_word --vocab-size 32000


In [ ]:
!python3 experiments/mamba_moe/ablate.py \
  --data-dir data/tok32k --word-data-dir data/tok32k_word \
  --out ablations \
  --configs none bottom ffn both ffn_pair ffn_pair_noshared ffn_pair_d128 ffn_pair_d256 ffn_pair_d512 local_ctx local_ctx8 local_ctx16 local_ctx32 local_ctx64 sandwich prior_gate retrieval_mm retrieval_multi retrieval_multi8 retrieval_suffix retrieval_knn hier_word \
  --steps 20 --batch 8 --seq-len 128 --d-model 128 --layers 2 --experts 64 --rank 8 \
  --shared-hidden 256 --eval-interval 10 --eval-batches 8 \
  --retrieval-tokens 100000 --retrieval-items 2000 --device cuda --threads 4


In [ ]:
import json, os
print(open('ablations/ablation.json').read())
print('files:', os.listdir('ablations')[:20])